# Lab Day 10 trên Kaggle: Qwen3-VL-4B nhận diện hư hại đường

Notebook này chạy E0, E1, E2, E3 và ablation E4 (LoRA rank 4/64) theo thứ tự. Sau khi chuẩn bị Kaggle theo KAGGLE_GUIDE.md, nhấn **Run all** một lần. Các ô sẽ tải dữ liệu và model, fine-tune, chấm điểm, lưu bằng chứng, vẽ 6 ảnh so sánh và tạo git commit local ở từng checkpoint.

Dataset và model được tải qua Internet; lần đầu có thể lâu. Adapter và dữ liệu tạm nằm trong work/ (đã gitignore), không commit trọng số. Các lệnh checkpoint chỉ commit report/evidence và ảnh minh họa.

## 0. Kaggle runtime và lấy mã nguồn

Trong Kaggle, bật Internet và GPU trong Notebook Settings, gắn dataset bằng Add Input trước khi chạy. Ô đầu sẽ clone repo mà bạn vừa push vào /kaggle/working. Không cần Kaggle API key hay Colab Secrets.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/dtc-z/K4-Track-4-Day10-VisionLanguageModel.git"
ROOT = Path("/kaggle/working/K4-Track-4-Day10-VisionLanguageModel")

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.run(["git", "config", "user.name", "Kaggle Lab Runner"], cwd=ROOT, check=True)
subprocess.run(["git", "config", "user.email", "kaggle-lab-runner@users.noreply.github.com"], cwd=ROOT, check=True)

print("Repo:", ROOT)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip())

Cài các thư viện đã ghim phiên bản; giữ nguyên torch/torchvision CUDA có sẵn trong ảnh Kaggle. Notebook dùng GPU đầu tiên (GPU 0) để tránh chia model giữa hai T4.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"],
    cwd=ROOT,
    check=True,
)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU. Vào Notebook Settings → Accelerator → GPU rồi khởi động lại.")
torch.cuda.set_device(0)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi", "-L"], check=True)

## 1. Dataset và chia split chống rò rỉ

Trước khi chạy, trong Kaggle bấm **Add Input**, tìm dataset **Road Damage Dataset - Potholes, Cracks and Manholes** của lorenzoarcioni rồi Add. Ô dưới tự tìm thư mục có cả images/ và labels-YOLO/, sau đó gọi nguyên scripts/prepare_data.py.

In [ ]:
from IPython.display import display
from scripts.labcommon import read_jsonl
from kaggle_lab import render_ground_truth, validate_split_sessions

INPUT_ROOT = Path("/kaggle/input")
if not INPUT_ROOT.exists():
    raise FileNotFoundError("/kaggle/input không tồn tại; hãy chạy notebook trong Kaggle.")

dataset_roots = sorted({
    image_dir.parent
    for image_dir in INPUT_ROOT.rglob("images")
    if image_dir.is_dir() and (image_dir.parent / "labels-YOLO").is_dir()
})
if len(dataset_roots) != 1:
    raise RuntimeError(
        "Cần gắn đúng một bản dataset có images/ và labels-YOLO/. "
        f"Tìm thấy: {[str(p) for p in dataset_roots]}"
    )
DATASET_ROOT = dataset_roots[0]
print("Dataset input:", DATASET_ROOT)

subprocess.run(
    [sys.executable, "scripts/prepare_data.py", "--source", str(DATASET_ROOT)],
    cwd=ROOT,
    check=True,
)
SPLITS_DIR = ROOT / "data" / "splits"
IMAGE_ROOT = ROOT / "data"
train_rows = read_jsonl(SPLITS_DIR / "train.jsonl")
val_rows = read_jsonl(SPLITS_DIR / "val.jsonl")
test_rows = read_jsonl(SPLITS_DIR / "test.jsonl")
split_rows = {"train": train_rows, "val": val_rows, "test": test_rows}
validate_split_sessions(split_rows)

dataset_meta = json.loads((SPLITS_DIR / "meta.json").read_text(encoding="utf-8"))
print("dataset_version:", dataset_meta["dataset_version"])
for split, rows in split_rows.items():
    boxes = {name: sum(obj["label"] == name for row in rows for obj in row["objects"])
             for name in ("pothole", "crack", "manhole")}
    print(f"{split}: {len(rows)} ảnh; boxes={boxes}")

for row in train_rows[:6]:
    print(row["image_id"], "objects:", len(row["objects"]))
    display(render_ground_truth(row, IMAGE_ROOT))

## 2. Nhãn YOLO → grounding JSON

Hàm chuyển đổi và hàm dựng câu trả lời đã được hiện thực trong kaggle_lab.py. Hãy kiểm tra vài nhãn/ảnh ở ô trước; mô hình học JSON list thuần với bbox_2d theo x1,y1,x2,y2.

In [ ]:
from kaggle_lab import objects_to_answer, yolo_to_box

sample = train_rows[0]
print("YOLO:", sample["objects"][:2])
print("Grounding JSON:", objects_to_answer(sample["objects"]))

## 3. E0: baseline zero-shot

Mỗi prediction lưu một dòng JSONL với image_id và raw output. score.py dùng cùng split test cố định, F1@IoU 0.5 và bootstrap CI95.

In [ ]:
from kaggle_lab import predict_split

def score_run(pred_path, name, evidence_path, meta=None, coord="norm1000"):
    command = [
        sys.executable, "scripts/score.py",
        "--pred", str(pred_path),
        "--name", name,
        "--out", str(evidence_path),
        "--coord", coord,
    ]
    for key, value in (meta or {}).items():
        command += ["--meta", f"{key}={value}"]
    subprocess.run(command, cwd=ROOT, check=True)
    return json.loads(Path(evidence_path).read_text(encoding="utf-8"))

def git_checkpoint(message, paths):
    paths = [str(path) for path in paths]
    subprocess.run(["git", "add", "--", *paths], cwd=ROOT, check=True)
    changed = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=ROOT)
    if changed.returncode == 0:
        print("Không có thay đổi mới để commit.")
        return
    if changed.returncode != 1:
        raise RuntimeError("Không kiểm tra được git index.")
    subprocess.run(["git", "commit", "-m", message], cwd=ROOT, check=True)
    print("Đã tạo local checkpoint commit:", message)

e0_pred = ROOT / "work" / "e0.jsonl"
e0_path = ROOT / "report" / "evidence" / "e0.json"
predict_split(test_rows, IMAGE_ROOT, e0_pred, max_new_tokens=256)
e0_result = score_run(e0_pred, "E0", e0_path)
print("E0 F1:", e0_result["micro"]["f1"], "parse:", e0_result["parse_valid_rate"])
git_checkpoint("lab: save E0 zero-shot evidence", [e0_path])

## 4. Chat template, mask loss và batch multimodal

encode_example() kiểm tra prompt là tiền tố của chuỗi hội thoại đầy đủ, rồi đặt labels=-100 cho toàn bộ user/image tokens. collate_examples() pad chuỗi văn bản và nối pixel_values/image_grid_thw theo chiều ảnh.

In [ ]:
from kaggle_lab import collate_examples, encode_example, load_processor

processor = load_processor()
sample_image = render_ground_truth(train_rows[0], IMAGE_ROOT)
sample_answer = objects_to_answer(train_rows[0]["objects"])
encoded = encode_example(sample_image, sample_answer, processor)
print("Input token shape:", tuple(encoded["input_ids"].shape))
print("Assistant target tokens:", int((encoded["labels"] != -100).sum()))
print("Target:", processor.tokenizer.decode(encoded["labels"][encoded["labels"] != -100]))
print("Image grid:", encoded.get("image_grid_thw"))
del processor, encoded, sample_image

## 5. LoRA / projector setup và 6. Huấn luyện E1–E3

Mỗi setup nạp lại model nền QLoRA 4-bit. LoRA target count được kiểm tra trong code (252 cho LLM, 348 cho LLM+ViT). E2/E3 mở merger và deepstack merger ở float32, checkpoint chúng riêng cạnh LoRA adapter.

In [ ]:
from kaggle_lab import train_sft

RUN_META = {}
RUN_RESULTS = {}
for setup in ("E1", "E2", "E3"):
    print("\n==========", setup, "==========")
    run_meta = train_sft(
        setup,
        train_rows,
        IMAGE_ROOT,
        output_dir=ROOT / "work",
        rank=16,
        alpha=32,
        max_steps=300,
        batch_size=2,
        grad_accum=4,
        learning_rate=2e-4,
        seed=42,
        checkpoint_every=100,
    )
    pred_path = ROOT / "work" / f"{setup.lower()}.jsonl"
    ev_path = ROOT / "report" / "evidence" / f"{setup.lower()}.json"
    predict_split(
        test_rows,
        IMAGE_ROOT,
        pred_path,
        setup=setup,
        output_dir=ROOT / "work",
        max_new_tokens=256,
    )
    result = score_run(
        pred_path,
        setup,
        ev_path,
        meta={
            "trainable_params": run_meta["trainable_params"],
            "peak_vram_gb": run_meta["peak_vram_gb"],
            "train_minutes": run_meta["train_minutes"],
        },
    )
    RUN_META[setup] = run_meta
    RUN_RESULTS[setup] = result
    print(setup, "F1:", result["micro"]["f1"], "CI95:", result["f1_ci95"])

    staged = [ev_path]
    if setup == "E1":
        from kaggle_lab import render_comparisons
        figures = render_comparisons(
            test_rows, pred_path, IMAGE_ROOT, ROOT / "report" / "figures", limit=6
        )
        staged += figures
    git_checkpoint(f"lab: save {setup} evidence", staged)

## 7. E1–E3 đã chấm; lưu so sánh trực quan

Ô huấn luyện phía trên sinh report/figures/ với 6 ảnh: box đúng màu đỏ, prediction màu xanh. Các file E0–E3 có kết quả thật và metadata runtime thật.

In [ ]:
for setup in ("E1", "E2", "E3"):
    result = RUN_RESULTS[setup]
    print(
        f"{setup}: F1={result['micro']['f1']:.4f}, "
        f"CI95={result['f1_ci95'][0]:.4f}–{result['f1_ci95'][1]:.4f}, "
        f"valid={result['parse_valid_rate']:.3f}, "
        f"FP/image={result['fp_per_image']:.3f}, "
        f"VRAM={RUN_META[setup]['peak_vram_gb']:.2f}GB, "
        f"train={RUN_META[setup]['train_minutes']:.1f}min"
    )

## 8. E4 ablation rank (4 và 64)

E4 giữ nguyên setup E1, seed, dữ liệu, prompt, optimizer, batch và số bước; chỉ thay rank. Hai biến thể được huấn luyện, chấm riêng và commit thành e4_rank_4.json, e4_rank_64.json.

In [ ]:
E4_RESULTS = {}
for rank in (4, 64):
    run_name = f"E4_rank_{rank}"
    print("\n==========", run_name, "==========")
    run_meta = train_sft(
        "E1",
        train_rows,
        IMAGE_ROOT,
        output_dir=ROOT / "work",
        checkpoint_name=run_name,
        rank=rank,
        alpha=2 * rank,
        max_steps=300,
        batch_size=2,
        grad_accum=4,
        learning_rate=2e-4,
        seed=42,
        checkpoint_every=100,
    )
    pred_path = ROOT / "work" / f"{run_name.lower()}.jsonl"
    ev_path = ROOT / "report" / "evidence" / f"e4_rank_{rank}.json"
    predict_split(
        test_rows,
        IMAGE_ROOT,
        pred_path,
        setup="E1",
        checkpoint_name=run_name,
        output_dir=ROOT / "work",
        rank=rank,
        max_new_tokens=256,
    )
    result = score_run(
        pred_path,
        f"E4-rank-{rank}",
        ev_path,
        meta={
            "trainable_params": run_meta["trainable_params"],
            "peak_vram_gb": run_meta["peak_vram_gb"],
            "train_minutes": run_meta["train_minutes"],
            "rank": rank,
        },
    )
    E4_RESULTS[rank] = result
    print(f"E4 rank={rank}: F1={result['micro']['f1']:.4f}, CI95={result['f1_ci95']}")
    git_checkpoint(f"lab: save E4 LoRA rank {rank} evidence", [ev_path])

## 9. E5 GRPO (bonus, tắt mặc định)

E5 khởi tạo từ adapter E1 và dùng reward IoU. Chạy phần này sau khi E0–E4 hoàn tất bằng cách đổi RUN_E5 thành True. E5 tốn thêm GPU/thời gian; do là bonus, nó không nằm trong mặc định một lần Run all.


In [ ]:
RUN_E5 = False
if RUN_E5:
    from kaggle_lab import compute_iou_reward, train_grpo_e5
    gt = [{"label": "pothole", "bbox": [400, 400, 600, 600]}]
    assert compute_iou_reward("[{\"label\":\"pothole\",\"bbox_2d\":[400,400,600,600]}]", gt) == 1.0
    assert compute_iou_reward("not json", gt) == 0.0
    assert compute_iou_reward("[{\"label\":\"crack\",\"bbox_2d\":[400,400,600,600]}]", gt) == 0.2
    extra = "[{\"label\":\"pothole\",\"bbox_2d\":[400,400,600,600]},{\"label\":\"pothole\",\"bbox_2d\":[0,0,100,100]}]"
    assert abs(compute_iou_reward(extra, gt) - 0.6) < 1e-8
    assert compute_iou_reward("[]", []) == 1.0
    e5_meta = train_grpo_e5(train_rows, IMAGE_ROOT, ROOT / "work", max_steps=60)
    e5_pred = ROOT / "work" / "e5.jsonl"
    e5_path = ROOT / "report" / "evidence" / "e5.json"
    predict_split(test_rows, IMAGE_ROOT, e5_pred, setup="E1", checkpoint_name="E5", output_dir=ROOT / "work")
    e5_result = score_run(e5_pred, "E5", e5_path, meta={
        "trainable_params": e5_meta["trainable_params"],
        "peak_vram_gb": e5_meta["peak_vram_gb"],
        "train_minutes": e5_meta["train_minutes"],
    })
    git_checkpoint("lab: save E5 GRPO evidence", [e5_path])
    print("E5 F1:", e5_result["micro"]["f1"], "CI95:", e5_result["f1_ci95"])


## 10. Hoàn tất và bước báo cáo

E5/GRPO là phần bonus tùy chọn theo GUIDE.md; notebook mặc định chạy đủ phần bắt buộc E0–E4. Kết quả, checkpoint commit và file cần điền trong REPORT.md được liệt kê bên dưới. Adapter, prediction JSONL và dữ liệu không được commit.

In [ ]:
from IPython.display import FileLink
bundle_path = ROOT / "day10-kaggle-results.bundle"
subprocess.run(["git", "bundle", "create", str(bundle_path), "--all"], cwd=ROOT, check=True)
print("Git bundle for downloading checkpoint history:", bundle_path)
display(FileLink("day10-kaggle-results.bundle"))

print("Evidence:")
for path in sorted((ROOT / "report" / "evidence").glob("*.json")):
    print(" -", path.relative_to(ROOT))
print("Figures:")
for path in sorted((ROOT / "report" / "figures").glob("*.jpg")):
    print(" -", path.relative_to(ROOT))
print("Local checkpoint history:")
print(subprocess.check_output(["git", "log", "-8", "--oneline"], cwd=ROOT, text=True))
print("dataset_version:", dataset_meta["dataset_version"])
print("E4 rank F1:", {rank: result["micro"]["f1"] for rank, result in E4_RESULTS.items()})
print("Tiếp theo: chép số liệu từ report/evidence/*.json vào REPORT.md, trả lời 10 câu bằng quan sát của bạn rồi commit REPORT.md.")